# 02 — Split Dataset (Idempotent)

Only run this if `01_setup_and_data_audit.ipynb` failed validation.

Calls `scripts/split_dataset.py` which:
- skips work if `dataset_resplit/` is already a valid 75/15/10 split
- re-splits from `dataset/` (legacy bad split or flat folder) otherwise
- copies files (does NOT move) — original `dataset/` stays intact

Splits are made **per-class** to guarantee every class is represented in every split.

## 1. Environment setup

This notebook is designed for **Google Colab (A100)**. The setup cell below:

1. Mounts Google Drive
2. Clones or updates the GitHub repo
3. Installs requirements
4. Adds the repo root to `sys.path` so `from src...` works

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
REPO_DIR = Path("/content/Image-Classification-with-CNN")

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping drive mount.")

# 2. Clone or pull
if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# 3. Make repo importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install requirements (Colab usually has torch already)
req = REPO_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

## 2. Decide source

The script supports two layouts:
- **Layout A:** `<src>/<class>/*.jpg` (flat — common after manual collection)
- **Layout B:** `<src>/<split>/<class>/*.jpg` (already split — we just re-pool and re-split)

The default `--src` is the legacy bad split on Drive. Change if your source is elsewhere.

In [ ]:
from pathlib import Path
from config.config import DATASET_DIR, DATASET_DIR_LEGACY

SRC = DATASET_DIR_LEGACY      
DST = DATASET_DIR              

print(f"Source: {SRC}  (exists: {SRC.exists()})")
print(f"Dest:   {DST}  (exists: {DST.exists()})")

## 3. Run the splitter

Pass `force=True` only if you need to overwrite an existing valid split (you usually don't).

In [ ]:
import subprocess, sys
force = False    
cmd = [
    sys.executable, "scripts/split_dataset.py",
    "--src", str(SRC),
    "--dst", str(DST),
    "--train", "0.75",
    "--val", "0.15",
    "--test", "0.10",
    "--seed", "42",
]
if force:
    cmd.append("--force")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"split_dataset.py failed with exit code {result.returncode}")

## 4. Quick verification

In [ ]:
from src.dataset import count_images_per_class
from config.config import CLASS_NAMES

counts = count_images_per_class(DST)
print(f"{'class':<32s} {'train':>6s} {'val':>6s} {'test':>6s}")
print("-" * 56)
for cls in CLASS_NAMES:
    tr = counts['train'].get(cls, 0)
    va = counts['val'].get(cls, 0)
    te = counts['test'].get(cls, 0)
    print(f"{cls:<32s} {tr:>6d} {va:>6d} {te:>6d}")

print()
print("✓ done. Re-run 01_setup_and_data_audit.ipynb to confirm validation passes.")